# 04 · Sentimento com CryptoBERT

Dos arquivos brutos até uma tabela diária de sentimento por moeda.
1. Limpa as notícias (cryptonews) e os tweets e separa os textos que citam cada moeda.
2. Dá uma nota a cada texto com o CryptoBERT: P(bullish) − P(bearish), de −1 a +1.
3. Faz a média do dia por fonte e preenche os dias sem texto de dois jeitos, usando só o passado:
   - **ffill:** repete o último valor da própria moeda.
   - **btc:** usa o valor do BTC no mesmo dia; se o BTC também não tiver, repete o último valor.
4. Confere se o sentimento acompanha o retorno de hoje e de amanhã.

In [ ]:
import os, sys, glob
achados = glob.glob('../src/tcc/config.py') + glob.glob('/kaggle/input/**/src/tcc/config.py', recursive=True)
RAIZ = achados[0].rsplit('/src/', 1)[0]              # pasta do projeto (no PC ou no dataset do Kaggle)
sys.path.insert(0, os.path.join(RAIZ, 'src'))
import warnings; warnings.filterwarnings('ignore')
import json, shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr
from tcc import config, dados, sentimento, textos

In [ ]:
INICIO = '2022-01-01'
FIM    = '2023-12-31'
MOEDAS = config.MOEDAS

SAIDA = config.PASTA_RESULTADOS / f'04_sentimento_cryptobert_{INICIO}_{FIM}'
METODOS = ['ffill', 'btc']
for metodo in METODOS:
    (SAIDA / f'sentimento_{metodo}').mkdir(parents=True, exist_ok=True)

## 1. Textos por moeda

In [ ]:
longa = textos.por_moeda(INICIO, FIM)
cobertura = textos.cobertura(longa, INICIO, FIM)
cobertura.round(1).to_csv(SAIDA / 'cobertura.csv', index=False)
print(longa['texto'].nunique(), 'textos diferentes')
cobertura.round(1)

## 2. CryptoBERT

In [ ]:
unicos = longa['texto'].drop_duplicates().tolist()
nota = dict(zip(unicos, sentimento.classificar(unicos)))
longa['nota'] = longa['texto'].map(nota)
longa.to_csv(SAIDA / 'nota_por_texto.csv', index=False)

In [ ]:
classe = pd.cut(longa['nota'], [-1, -1/3, 1/3, 1], labels=['bearish', 'neutro', 'bullish'], include_lowest=True)
distribuicao = pd.crosstab(longa['fonte'], classe, normalize='index') * 100
distribuicao['nota_media'] = longa.groupby('fonte')['nota'].mean()
distribuicao.round(3).to_csv(SAIDA / 'distribuicao_notas.csv')

fig, ax = plt.subplots(figsize=(8, 3))
for fonte, g in longa.drop_duplicates(['fonte', 'texto']).groupby('fonte'):
    ax.hist(g['nota'], bins=40, alpha=.6, density=True, label=fonte)
ax.set_xlabel('nota CryptoBERT'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(SAIDA / 'distribuicao_notas.png', dpi=130); plt.show()
distribuicao.round(2)

## 3. Tabela diária

In [ ]:
brutos = {m: sentimento.diario(longa, m, INICIO, FIM) for m in MOEDAS}
resumo = []
for metodo in METODOS:
    for m in MOEDAS:
        base = brutos['BTC'] if metodo == 'btc' and m != 'BTC' else None      # o BTC só usa o ffill
        final = sentimento.preencher(brutos[m], btc=base)
        final.round(6).to_csv(SAIDA / f'sentimento_{metodo}' / f'{m}_sentimento.csv')
        linha = {'metodo': metodo, 'moeda': m, 'dias': len(final)}
        for nome in ['Tweet', 'Noticia']:
            linha[f'dias_proprios_{nome}'] = final[f'Tem{nome}'].sum()
            linha[f'dias_do_btc_{nome}'] = final[f'CopiaBTC{nome}'].sum() if f'CopiaBTC{nome}' in final else 0
            linha[f'dias_ffill_{nome}'] = len(final) - linha[f'dias_proprios_{nome}'] - linha[f'dias_do_btc_{nome}']
        resumo.append(linha)
resumo = pd.DataFrame(resumo)
resumo.to_csv(SAIDA / 'resumo.csv', index=False)
resumo

In [ ]:
fig, eixos = plt.subplots(len(MOEDAS), 1, figsize=(11, 2 * len(MOEDAS)), sharex=True)
for ax, m in zip(eixos, MOEDAS):
    for col in sentimento.FONTES.values():
        ax.plot(brutos[m][col].rolling(7, min_periods=1).mean(), lw=1, label=col)
    ax.set_ylabel(m); ax.grid(alpha=.3)
eixos[0].legend(ncol=2); eixos[0].set_title('Sentimento diário (média móvel de 7 dias, só dias com texto)')
plt.tight_layout(); plt.savefig(SAIDA / 'sentimento_diario.png', dpi=130); plt.show()

## 4. Sentimento x retorno

Spearman nos dias com texto. Para ajudar a prever, o sentimento de hoje precisa acompanhar o retorno de **amanhã**.

In [ ]:
linhas = []
for m in MOEDAS:
    retorno = 100 * np.log(dados.precos(m, INICIO, FIM)['Close']).diff()
    for col in sentimento.FONTES.values():
        s = brutos[m][col]
        for quando, r in [('hoje', retorno), ('amanha', retorno.shift(-1))]:
            ok = s.notna() & r.notna()
            rho, p = spearmanr(s[ok], r[ok])
            linhas.append({'moeda': m, 'fonte': col, 'retorno': quando, 'rho': rho, 'p': p, 'dias': int(ok.sum())})
correlacao = pd.DataFrame(linhas)
correlacao.round(4).to_csv(SAIDA / 'correlacao_retorno.csv', index=False)
correlacao.pivot_table(index=['moeda', 'fonte'], columns='retorno', values=['rho', 'p']).round(3)

In [ ]:
(SAIDA / 'config.json').write_text(json.dumps({'inicio': INICIO, 'fim': FIM, 'moedas': MOEDAS,
    'modelo': config.MODELO_SENTIMENTO, 'max_tokens': config.MAX_TOKENS, 'padroes_moeda': config.PADROES_MOEDA,
    'arquivos': [config.ARQ_NOTICIAS, config.ARQ_TWEETS]}, indent=2))
print('Pronto:', shutil.make_archive(str(SAIDA), 'zip', SAIDA))